# NLP Text Preprocessing & Text Vectorisation

Machine-learning models need numbers, but text is just strings. In this notebook I practise the steps that turn raw text (hotel reviews) into numeric features.

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

# Download NLTK resources (run once)
nltk.download('punkt')
nltk.download('punkt_tab') # - ensure this is installed
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Sample travel review data
data = {
    "Rating": [5, 2, 4, 5],
    "Price": [120, 95, 200, 150],
    "Review": [
        "Great location, very clean!",
        "Not worth the price.",
        "Will definitely stay again.",
        "This hotel was the bomb!!!"
    ]
}

df = pd.DataFrame(data)
print("Original Data:")
print(df)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


Original Data:
   Rating  Price                       Review
0       5    120  Great location, very clean!
1       2     95         Not worth the price.
2       4    200  Will definitely stay again.
3       5    150   This hotel was the bomb!!!


## 1. Why text needs preprocessing
I can take the mean of `Rating` and `Price`, but `mean()` on the `Review` column fails. Pandas tries to join the strings and convert them into a number. Text has to be converted into numbers first.

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
print(df)

# Calculate the average hotel rating
average_rating = df['Rating'].mean()
print("Average rating:", average_rating)

# Calculate the average price
average_price = df['Price'].mean()
print("Average price:", average_price)

# Try to calculate the 'average' review (doesn't work!)
try:
    print(df['Review'].mean())
except Exception as e:
    print("Error when trying to calculate mean of reviews:", e)

   Rating  Price                       Review
0       5    120  Great location, very clean!
1       2     95         Not worth the price.
2       4    200  Will definitely stay again.
3       5    150   This hotel was the bomb!!!
Average rating: 4.0
Average price: 141.25
Error when trying to calculate mean of reviews: Could not convert string 'Great location, very clean!Not worth the price.Will definitely stay again.This hotel was the bomb!!!' to numeric


## 2. Tokenisation
- `word_tokenize` splits text into words and punctuation. It's smart enough to keep "Mr." and "IIT-Delhi" together.
- `sent_tokenize` splits text into sentences and doesn't break at the full stop in "Mr.".

In [ ]:
import nltk
nltk.download('punkt') # required for tokenizer
nltk.download('punkt_tab') # required for newer NLTK tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

# Split text into words using NLTK
text = "Mr. Gyansetu graduated from IIT-Delhi. He later started an analytics firm called Lux, which catered to enterprise customers."
words = word_tokenize(text)
print(words)

# Split text into sentences
text = "Mr. Gyansetu graduated from IIT-Delhi. He later started an analytics firm called Lux, which catered to enterprise customers."
sentences = sent_tokenize(text)
print(sentences)

['Mr.', 'Gyansetu', 'graduated', 'from', 'IIT-Delhi', '.', 'He', 'later', 'started', 'an', 'analytics', 'firm', 'called', 'Lux', ',', 'which', 'catered', 'to', 'enterprise', 'customers', '.']
['Mr. Gyansetu graduated from IIT-Delhi.', 'He later started an analytics firm called Lux, which catered to enterprise customers.']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 3. Stop-word removal
Stop words ("the", "from", "an", "which") are very common and carry little meaning, so removing them reduces noise.

**But be careful:** NLTK's list includes **"not"**. In the review "Not worth the price", removing "not" flips the meaning. For sentiment analysis I'd keep negation words.

In [ ]:
import nltk
nltk.download('stopwords') #corpora of stopwords (required for stopwords)
  # List of stop words
from nltk.corpus import stopwords
print(stopwords.words("english"))

# Remove stop words
words = [w for w in words if w not in stopwords.words("english")]
print(words)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 4. Part-of-speech tagging & named entities
- **POS tagging** uses context. In "I always lie down to tell a lie", the first *lie* is tagged as a verb (VBP) and the second as a noun (NN).
- **Named-entity recognition** finds Peter (PERSON), InfoTech (ORGANIZATION) and India (GPE = location).

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
from nltk import pos_tag
from nltk.tokenize import word_tokenize
# Tag parts of speech (PoS)
sentence = word_tokenize("I always lie down to tell a lie.")
pos_tag(sentence)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


[('I', 'PRP'),
 ('always', 'RB'),
 ('lie', 'VBP'),
 ('down', 'RP'),
 ('to', 'TO'),
 ('tell', 'VB'),
 ('a', 'DT'),
 ('lie', 'NN'),
 ('.', '.')]

In [ ]:
import nltk
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
from nltk import pos_tag, ne_chunk
from nltk.tokenize import word_tokenize
# Recognize named entities in a tagged sentence
out = ne_chunk(pos_tag(word_tokenize("Peter joined InfoTech Ltd. in India.")))
print(out.__repr__())

[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


Tree('S', [Tree('PERSON', [('Peter', 'NNP')]), ('joined', 'VBD'), Tree('ORGANIZATION', [('InfoTech', 'NNP')]), ('Ltd.', 'NNP'), ('in', 'IN'), Tree('GPE', [('India', 'NNP')]), ('.', '.')])


## 5. Full cleaning pipeline
1. Lowercase
2. Remove punctuation and emojis
3. Tokenise
4. Remove stop words
5. Lemmatise (reduce words to their base form)

**What I noticed:** "Not worth the price" became just `[worth, price]`. The negative meaning is lost, which is the stop-word problem from section 3. Slang like "the bomb" also stays as `bomb`, which a model might read as negative even though it's praise.

In [ ]:
# Step 1: Lowercase conversion (standardization)
df['clean_review'] = df['Review'].str.lower()

# Step 2: Remove punctuation and special characters (noise removal)
def remove_noise(text):
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = re.sub(r'[^\x00-\x7F]+', '', text) # Remove non-ASCII (e.g., emojis)
    return text

df['clean_review'] = df['clean_review'].apply(remove_noise)

# Step 3: Tokenization (split into words)
df['tokens'] = df['clean_review'].apply(nltk.word_tokenize)

# Step 4: Remove stopwords (common uninformative words)
stop_words = set(stopwords.words('english'))
df['tokens_no_stop'] = df['tokens'].apply(
    lambda tokens: [w for w in tokens if w not in stop_words]
)

# Step 5: Lemmatization (reduce words to base form)
lemmatizer = WordNetLemmatizer()
df['lemmatized'] = df['tokens_no_stop'].apply(
    lambda tokens: [lemmatizer.lemmatize(t) for t in tokens]
)

# Display the processed DataFrame columns
print("\nProcessed Text Data:")
print(df[['Review', 'clean_review', 'tokens', 'tokens_no_stop', 'lemmatized']])


Processed Text Data:
                        Review                clean_review  \
0  Great location, very clean!   great location very clean   
1         Not worth the price.         not worth the price   
2  Will definitely stay again.  will definitely stay again   
3   This hotel was the bomb!!!     this hotel was the bomb   

                            tokens            tokens_no_stop  \
0   [great, location, very, clean]  [great, location, clean]   
1         [not, worth, the, price]            [worth, price]   
2  [will, definitely, stay, again]        [definitely, stay]   
3    [this, hotel, was, the, bomb]             [hotel, bomb]   

                 lemmatized  
0  [great, location, clean]  
1            [worth, price]  
2        [definitely, stay]  
3             [hotel, bomb]  


## 6. Turning text into numbers: Bag-of-Words vs TF-IDF
- **CountVectorizer (bag-of-words):** each column is a word, and each value is how many times it appears.
- **TF-IDF:** term frequency × inverse document frequency. A word that appears in **many** documents (like "coding" or "love") gets a **lower** weight. A word that is unique to one document gets a higher weight.

In the output, "coding" gets a lower score in sentence 2 (0.47) than "fun" (0.62), because "coding" also appears in sentence 1.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Sample documents
documents = [
    "I love coding.",
    "Coding is fun.",
    "I love learning new things."
]

# Using CountVectorizer
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(documents)
print("Count Vectorizer Result:\n", count_matrix.toarray())

# Using TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
print("TF-IDF Vectorizer Result:\n", tfidf_matrix.toarray())

Count Vectorizer Result:
 [[1 0 0 0 1 0 0]
 [1 1 1 0 0 0 0]
 [0 0 0 1 1 1 1]]
TF-IDF Vectorizer Result:
 [[0.70710678 0.         0.         0.         0.70710678 0.
  0.        ]
 [0.4736296  0.62276601 0.62276601 0.         0.         0.
  0.        ]
 [0.         0.         0.         0.52863461 0.40204024 0.52863461
  0.52863461]]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define a small sample corpus (a list of documents)
documents = [
    "Natural language processing and machine learning are fascinating fields.",
    "Machine learning models require clean training data.",
    "Natural language processing involves text preprocessing and feature extraction."
]

# 2. Initialize the TfidfVectorizer
vectorizer = TfidfVectorizer()

# 3. Learn vocabulary and compute TF-IDF scores
tfidf_matrix = vectorizer.fit_transform(documents)

# 4. Extract the vocabulary terms (features)
feature_names = vectorizer.get_feature_names_out()

# 5. Display the results
print("Vocabulary / Terms:")
print(feature_names)

print("\nTF-IDF Matrix (Rows = Documents, Columns = Terms):")
print(tfidf_matrix.toarray())

Vocabulary / Terms:
['and' 'are' 'clean' 'data' 'extraction' 'fascinating' 'feature' 'fields'
 'involves' 'language' 'learning' 'machine' 'models' 'natural'
 'preprocessing' 'processing' 'require' 'text' 'training']

TF-IDF Matrix (Rows = Documents, Columns = Terms):
[[0.29898437 0.39312851 0.         0.         0.         0.39312851
  0.         0.39312851 0.         0.29898437 0.29898437 0.29898437
  0.         0.29898437 0.         0.29898437 0.         0.
  0.        ]
 [0.         0.         0.40301621 0.40301621 0.         0.
  0.         0.         0.         0.         0.30650422 0.30650422
  0.40301621 0.         0.         0.         0.40301621 0.
  0.40301621]
 [0.28122142 0.         0.         0.         0.36977238 0.
  0.36977238 0.         0.36977238 0.28122142 0.         0.
  0.         0.28122142 0.36977238 0.28122142 0.         0.36977238
  0.        ]]


## What I learned
- Every preprocessing step is a choice that can remove meaning (e.g. removing "not").
- Lemmatisation gives real dictionary words. It's slower than stemming but more accurate.
- TF-IDF highlights the words that make each document different. Common words are down-weighted automatically.